# Milestone 5: Merging Data, Database Storage & Visualization
## Author: Karthikeya Allada
## Date: 05/30/2026
## Course: DSC540-T302 Data Preparation (2265-1)
## Global Quality of Life Analysis

**Database:** SQLite (file-based: quality_of_life.db)  
**Data Sources:**
1. **Flat File (CSV):** World Happiness Report Kaggle (`jainaru/world-happiness-report-2024-yearly-updated`)  cleaned in Milestone 2
2. **Website (HTML):** Wikipedia List of Countries by GDP (Nominal)  cleaned in Milestone 3
3. **API (JSON):** REST Countries API v3.1  cleaned in Milestone 4

This notebook  loads each cleaned dataset into SQLite as individual tables, joins them using SQL into one consolidated dataset, and produces 5 labeled visualizations. At least 2 visualizations use data from more than one source table via SQL joins.

**Setup & Imports**
**Imports all required Python libraries and sets display/plotting defaults.Establishes a consistent environment so data cleaning, SQL operations, and visualizations run reproducibly.**

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import requests
import json
import re
from datetime import datetime
from bs4 import BeautifulSoup
from thefuzz import process, fuzz
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
import os, glob
import pandas as pd
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 200)
sns.set_theme(style='whitegrid', font_scale=1.1)

print("All libraries loaded successfully.")

**World Happiness Report (WHR)**
**DataSouce: Flat File / CSV)**

**Source: Kaggle dataset jainaru/world-happiness-report-2024-yearly-updated - pulled live via kaggle website
7 Cleaning Steps: Rename headers, fix special characters & casing, add regional indicator via merge, flag IQR outliers, impute missing values (per-country then regional median fallback), filter to most recent year per country, round numeric columns.**

**Downloads the World Happiness Report files from Kaggle and loads the raw CSVs into DataFrames.**
**Brings the source data into the notebook so the full cleaning pipeline can be rerun end to end.**

In [ ]:
# Install/upgrade kagglehub and its SDK dependency in user site-packages
# Install/upgrade notebook dependencies in user site-packages
import site
import subprocess
import sys

print('Installing/upgrading kagglehub + kagglesdk + kaleido with --user...')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--user', '--upgrade',
    'kagglehub', 'kagglesdk', 'kaleido'
])

site.addsitedir(site.getusersitepackages())

# Compatibility shim for some kagglesdk/kagglehub builds
import kagglesdk.kaggle_env as kaggle_env
if not hasattr(kaggle_env, 'get_web_endpoint'):
    kaggle_env.get_web_endpoint = lambda *args, **kwargs: 'https://www.kaggle.com'

import kagglehub
import kagglehub.handle as kh_handle
kh_handle.get_web_endpoint = lambda *args, **kwargs: 'https://www.kaggle.com'
print('kagglehub import successful.')

import kaleido
print('kaleido import successful.')

In [ ]:
import kagglehub
import kagglehub.handle as kh_handle
import kagglesdk.kaggle_env as kaggle_env

# Compatibility shim for some Python 3.13 kagglehub/kagglesdk combinations
if not hasattr(kaggle_env, 'get_web_endpoint') or callable(getattr(kaggle_env, 'get_web_endpoint', None)) and kaggle_env.get_web_endpoint.__code__.co_argcount == 0:
    kaggle_env.get_web_endpoint = lambda *args, **kwargs: 'https://www.kaggle.com'
kh_handle.get_web_endpoint = lambda *args, **kwargs: 'https://www.kaggle.com'

# Download the dataset from Kaggle
dataset = "jainaru/world-happiness-report-2024-yearly-updated"
print(f"Downloading dataset: {dataset}")
path = kagglehub.dataset_download(dataset)
print(f"\u2714 Downloaded to: {path}")

# Locate the CSV files in the download folder
csv_files = glob.glob(os.path.join(path, '*.csv'))
print(f"  Files found: {[os.path.basename(f) for f in csv_files]}")

# Identify the two files
whr_file = [f for f in csv_files if 'updated' in os.path.basename(f).lower()][0]
snapshot_file = [f for f in csv_files if 'updated' not in os.path.basename(f).lower() and '2024' in os.path.basename(f)][0]

def read_csv_with_fallback(file_path):
    try:
        return pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding='latin1')

df_whr_raw = read_csv_with_fallback(whr_file)
df_snapshot = read_csv_with_fallback(snapshot_file)

print(f"\n\u2714 WHR multi-year file loaded: {df_whr_raw.shape[0]} rows x {df_whr_raw.shape[1]} cols")
print(f"  File: {os.path.basename(whr_file)}")
print(f"\u2714 WHR snapshot file loaded: {df_snapshot.shape[0]} rows x {df_snapshot.shape[1]} cols")
print(f"  File: {os.path.basename(snapshot_file)}")

**Applies all cleaning steps to WHR data (renaming, normalization, imputation, outlier flagging, and final shaping).Produces a standardized, analysis-ready happiness table with consistent country keys for later SQL joins.**

In [ ]:
df = df_whr_raw.copy()

# Step 1: Rename headers to snake_case
rename_map = {
    'Country name': 'country',
    'year': 'year',
    'Life Ladder': 'happiness_score',
    'Log GDP per capita': 'log_gdp_per_capita',
    'Social support': 'social_support',
    'Healthy life expectancy at birth': 'healthy_life_expectancy',
    'Freedom to make life choices': 'freedom_score',
    'Generosity': 'generosity',
    'Perceptions of corruption': 'corruption_perception',
    'Positive affect': 'positive_affect',
    'Negative affect': 'negative_affect'
}
df.rename(columns=rename_map, inplace=True)
print(f"Step 1: Renamed {len(rename_map)} columns to snake_case")

# Step 2: Fix special characters (non-ASCII) and apply Title Case
df['country'] = (df['country']
    .str.replace(r'[^\x00-\x7F]', '', regex=True)
    .str.strip()
    .str.title())
print(f"Step 2: Fixed non-ASCII chars, applied Title Case to {df['country'].nunique()} country names")

# Step 3: Add Regional Indicator via merge with snapshot file
region_map = df_snapshot[['Country name', 'Regional indicator']].drop_duplicates()
region_map.columns = ['country', 'region']
region_map['country'] = (region_map['country']
    .str.replace(r'[^\x00-\x7F]', '', regex=True)
    .str.strip()
    .str.title())
df = df.merge(region_map, on='country', how='left')
df['region'] = df['region'].fillna('Unclassified')
print(f"Step 3: Merged Regional Indicator; {(df['region']=='Unclassified').sum()} unmatched set to 'Unclassified'")

# Step 4: Flag IQR outliers on happiness_score
Q1 = df['happiness_score'].quantile(0.25)
Q3 = df['happiness_score'].quantile(0.75)
IQR = Q3 - Q1
df['is_outlier'] = ((df['happiness_score'] < Q1 - 1.5 * IQR) |
                    (df['happiness_score'] > Q3 + 1.5 * IQR))
outlier_count = df['is_outlier'].sum()
print(f"Step 4: IQR outlier detection - {outlier_count} rows flagged")

# Step 5: Impute missing values (per-country median, then regional median fallback)
numeric_cols = ['happiness_score', 'log_gdp_per_capita', 'social_support',
                'healthy_life_expectancy', 'freedom_score', 'generosity',
                'corruption_perception', 'positive_affect', 'negative_affect']
for col in numeric_cols:
    df[col] = df.groupby('country')[col].transform(lambda x: x.fillna(x.median()))
    if df[col].isna().sum() > 0:
        regional_medians = df.groupby('region')[col].transform('median')
        df[col] = df[col].fillna(regional_medians)
print(f"Step 5: Imputed missing values using per-country then regional median fallback")

# Step 6: Filter to most recent year per country
df_history = df.copy()  # Preserve full history
idx = df.groupby('country')['year'].idxmax()
df = df.loc[idx].reset_index(drop=True)
print(f"Step 6: Filtered {len(df_history)} historical rows to {df.shape[0]} (most recent year per country)")

# Step 7: Round numeric columns, cast year to int
for col in numeric_cols:
    df[col] = df[col].round(3)
df['year'] = df['year'].astype(int)
print(f"Step 7: Rounded all float columns to 3 d.p.; cast year to int")

# Final column order matching Milestone 2
col_order = ['country', 'year', 'happiness_score', 'log_gdp_per_capita',
             'social_support', 'healthy_life_expectancy', 'freedom_score',
             'generosity', 'corruption_perception', 'positive_affect',
             'negative_affect', 'region', 'is_outlier']
df_whr = df[col_order].sort_values('happiness_score', ascending=False).reset_index(drop=True)

print(f"\n\u2714 WHR cleaned: {df_whr.shape[0]} countries Ã— {df_whr.shape[1]} columns")
print(f"  Columns: {list(df_whr.columns)}")
df_whr.head()


## Wikipedia GDP Data (Website / HTML Scraping)
## Source: [Wikipedia - List of Countries by GDP (Nominal)](https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal))  
## 8 Cleaning Steps: Standardize headers, clean GDP values, clean country names (remove annotations), fuzzy matching, missing data & outliers, duplicate detection, add calculated fields, final validation.
**Sends an HTTP request to Wikipedia, parses HTML, and extracts the target GDP table into a raw DataFrame.Captures website data live so GDP information can be cleaned and merged with the other sources.**

In [ ]:
# Scrape Wikipedia GDP table live
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'
print(f"Fetching data from: {url}")

headers = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/124.0.0.0 Safari/537.36'),
    'Accept-Language': 'en-US,en;q=0.9'
}

try:
    response = requests.get(url, headers=headers, timeout=15)
    html_content = response.text
    if response.status_code != 200:
        raise ValueError(f"HTTP {response.status_code}")
    if 'wikitable' not in html_content.lower():
        raise ValueError("No wikitable found in response")
    print(f"\u2714 HTML fetched successfully - Status: {response.status_code}")
    print(f"  Content length: {len(html_content):,} characters")
except Exception as e:
    print(f"\u2718 Scraping failed: {e}")
    raise

# Parse HTML and extract the first wikitable
soup = BeautifulSoup(html_content, 'html.parser')
tables = soup.find_all('table', {'class': 'wikitable'})
if not tables:
    tables = soup.find_all('table', {'class': 'sortable'})

target_table = tables[0] if tables else None

rows_data = []
for tr in target_table.find_all('tr')[1:]:
    cells_in_row = tr.find_all(['td', 'th'])
    if len(cells_in_row) >= 2:
        row = [cell.get_text(strip=True) for cell in cells_in_row]
        rows_data.append(row)

df_raw = pd.DataFrame(rows_data[:60])
print(f"  Raw table extracted: {df_raw.shape[0]} rows Ã— {df_raw.shape[1]} cols")

**Below code cleans and validates the scraped GDP dataset (header fixes, numeric conversion, name cleanup, duplicate handling, and derived fields).Converts messy web-scraped text into reliable numeric GDP features usable for analysis and joining.**

In [ ]:
#Steps 1â€“8: Clean the Wikipedia GDP data (Milestone 3 logic)
df_gdp = df_raw.copy()

# Step 1: Replace and standardize headers
df_gdp.columns = ['Country_Territory', 'IMF_GDP', 'WorldBank_GDP', 'UN_GDP']
if len(df_gdp) > 0 and df_gdp.iloc[0]['Country_Territory'] == 'World':
    df_gdp = df_gdp.iloc[1:].reset_index(drop=True)
df_gdp.insert(0, 'Rank', range(1, len(df_gdp) + 1))
print(f"Step 1: Headers standardized - {list(df_gdp.columns)}")

# Step 2: Clean GDP values (remove formatting, convert to numeric)
def clean_gdp_value(value):
    if pd.isna(value) or value == '' or value == '-':
        return None
    cleaned = re.sub(r'[,$\xa0-]', '', str(value))
    cleaned = re.sub(r'[^0-9.-]', '', cleaned)
    try:
        return float(cleaned) if cleaned else None
    except ValueError:
        return None

df_gdp['GDP_Million_USD'] = df_gdp['IMF_GDP'].apply(clean_gdp_value)
print(f"Step 2: GDP values cleaned to numeric (IMF as primary source)")

# Step 3: Clean country names (remove annotations, footnotes)
df_gdp['Country_Territory'] = (df_gdp['Country_Territory']
    .astype(str)
    .str.replace(r'\[.*?\]', '', regex=True)
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.replace(r'[*â€ â€¡Â§]', '', regex=True)
    .str.replace(r'\xa0', ' ', regex=True)
    .str.strip()
    .str.title())
print(f"Step 3: Country names cleaned - annotations/footnotes removed")

# Step 4: Fuzzy matching analysis (internal consistency check)
countries = df_gdp['Country_Territory'].dropna().unique().tolist()
similar_pairs = []
checked_pairs = set()
for i, c1 in enumerate(countries):
    for c2 in countries[i+1:]:
        pair_key = tuple(sorted([c1, c2]))
        if pair_key not in checked_pairs:
            best_score = max(fuzz.ratio(c1.lower(), c2.lower()),
                           fuzz.partial_ratio(c1.lower(), c2.lower()),
                           fuzz.token_sort_ratio(c1.lower(), c2.lower()))
            if 80 < best_score < 100:
                similar_pairs.append({'Country_1': c1, 'Country_2': c2, 'Score': best_score})
                checked_pairs.add(pair_key)
print(f"Step 4: Fuzzy matching - {len(similar_pairs)} similar pairs detected")

# Step 5: Handle missing data and identify outliers
missing_gdp = df_gdp['GDP_Million_USD'].isna().sum()
df_cleaned = df_gdp.dropna(subset=['GDP_Million_USD']).copy()
print(f"Step 5: Removed {missing_gdp} rows with missing GDP - {len(df_cleaned)} remain")

# Step 6: Remove duplicate entries
dupes_before = df_cleaned.duplicated(subset=['Country_Territory']).sum()
df_final_gdp = df_cleaned.drop_duplicates(subset=['Country_Territory'], keep='first').copy()
print(f"Step 6: Duplicate check - {dupes_before} duplicates removed")

# Step 7: Add calculated fields
df_final_gdp['GDP_Billion_USD'] = df_final_gdp['GDP_Million_USD'] / 1000

def categorize_economy(gdp_billion):
    if gdp_billion >= 1000:
        return 'Large Economy (>$1T)'
    elif gdp_billion >= 100:
        return 'Medium Economy ($100B-$1T)'
    else:
        return 'Small Economy (<$100B)'

df_final_gdp['Economy_Category'] = df_final_gdp['GDP_Billion_USD'].apply(categorize_economy)
df_final_gdp['GDP_Formatted'] = df_final_gdp['GDP_Billion_USD'].apply(lambda x: f"${x:,.2f}B")
df_final_gdp = df_final_gdp.reset_index(drop=True)
print(f"Step 7: Added GDP_Billion_USD, Economy_Category, GDP_Formatted")

# Step 8: Final validation
total_missing = df_final_gdp.isna().sum().sum()
print(f"Step 8: Validation - {total_missing} missing values in final dataset")

# Re-rank after cleaning
df_final_gdp['Rank'] = range(1, len(df_final_gdp) + 1)

print(f"\n\u2714 Wikipedia GDP cleaned: {df_final_gdp.shape[0]} countries Ã— {df_final_gdp.shape[1]} columns")
print(f"  Columns: {list(df_final_gdp.columns)}")
df_final_gdp.head()


## REST Countries API (JSON)
**Source: [REST Countries API v3.1](https://restcountries.com/)  
7 Cleaning Steps: Flatten nested JSON â†’ standardize headers â†’ fix casing â†’ handle missing data â†’ IQR outlier flags â†’ remove duplicates â†’ fuzzy matching against WHR names â†’ format for readability.**
**Below code calls the REST Countries API and flattens nested JSON fields into a tabular DataFrame.Normalizes API response structure so country metadata can be cleaned and merged with WHR and GDP data.**

In [ ]:
#Pull data from REST Countries API (reproducing Milestone 4)
api_url = "https://restcountries.com/v3.1/all?fields=name,capital,region,subregion,population,area,languages,currencies,latlng,timezones"

try:
    resp = requests.get(api_url, timeout=30)
    resp.raise_for_status()
    raw_json = resp.json()
    print(f"\u2714 REST Countries API: {len(raw_json)} country records received")
except Exception as e:
    print(f"\u2718 API call failed: {e}")
    raise

# Flatten nested JSON into rows (exact Milestone 4 logic)
rows = []
for country in raw_json:
    name_info = country.get('name', {})
    common_name = name_info.get('common', '')
    official_name = name_info.get('official', '')
    capitals = country.get('capital', [])
    capital = capitals[0] if capitals else None
    region = country.get('region', None)
    subregion = country.get('subregion', None)
    population = country.get('population', None)
    area = country.get('area', None)
    langs = country.get('languages', {})
    primary_language = list(langs.values())[0] if langs else None
    all_languages = ', '.join(langs.values()) if langs else None
    num_languages = len(langs)
    currs = country.get('currencies', {})
    if currs:
        first_key = list(currs.keys())[0]
        first_curr = currs[first_key]
        curr_code = first_key
        curr_name = first_curr.get('name', None)
        curr_symbol = first_curr.get('symbol', None)
    else:
        curr_code = curr_name = curr_symbol = None
    latlng = country.get('latlng', [None, None])
    tz = country.get('timezones', [])

    rows.append({
        'Country_Name': common_name,
        'Official_Name': official_name,
        'Capital': capital,
        'Region': region,
        'Subregion': subregion,
        'Population': population,
        'Area_sq_km': area,
        'Primary_Language': primary_language,
        'All_Languages': all_languages,
        'Num_Languages': num_languages,
        'Currency_Code': curr_code,
        'Currency_Name': curr_name,
        'Currency_Symbol': curr_symbol,
        'Latitude': latlng[0] if len(latlng) > 0 else None,
        'Longitude': latlng[1] if len(latlng) > 1 else None,
        'Primary_Timezone': tz[0] if tz else None,
        'Num_Timezones': len(tz),
    })

df_api = pd.DataFrame(rows)
print(f"  Flattened DataFrame: {df_api.shape[0]} rows Ã— {df_api.shape[1]} columns")


**Below code executes Milestone 4 cleaning rules on API data, including standardization, missing-data handling, outlier flags, fuzzy matching, and derived metrics.Aligns API country records with other datasets and improves data quality for accurate integration.**

In [ ]:
#Cleaning Steps (1â€“7)
# Step 1: Replace headers (standardize naming)
header_map = {
    'Country_Name': 'Country',
    'Official_Name': 'Official_Name',
    'Capital': 'Capital_City',
    'Region': 'Continent',
    'Subregion': 'Subregion',
    'Population': 'Population',
    'Area_sq_km': 'Area_Sq_Km',
    'Primary_Language': 'Primary_Language',
    'All_Languages': 'All_Languages',
    'Num_Languages': 'Language_Count',
    'Currency_Code': 'Currency_Code',
    'Currency_Name': 'Currency_Name',
    'Currency_Symbol': 'Currency_Symbol',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude',
    'Primary_Timezone': 'Primary_Timezone',
    'Num_Timezones': 'Timezone_Count',
}
df_api.rename(columns=header_map, inplace=True)
print(f"Step 1: Headers standardized - {list(df_api.columns)}")

# Step 2: Fix casing and inconsistent country names
df_api['Country'] = df_api['Country'].str.strip().str.title()
df_api['Official_Name'] = df_api['Official_Name'].str.strip()
df_api['Capital_City'] = df_api['Capital_City'].str.strip().str.title()
df_api['Continent'] = df_api['Continent'].str.strip().str.title()
df_api['Subregion'] = df_api['Subregion'].str.strip().str.title()
case_fixes = {
    'United States Of America': 'United States',
    "CÃ´te D'Ivoire": "CÃ´te d'Ivoire",
}
df_api['Country'] = df_api['Country'].replace(case_fixes)
print(f"Step 2: Casing standardized for Country, Capital, Continent, Subregion")

# Step 3: Handle missing data
text_cols = ['Capital_City', 'Primary_Language', 'All_Languages',
             'Currency_Code', 'Currency_Name', 'Currency_Symbol',
             'Primary_Timezone', 'Subregion']
for col in text_cols:
    df_api[col] = df_api[col].fillna('Unknown')
df_api['Data_Quality_Flag'] = 'OK'
df_api.loc[df_api['Population'] == 0, 'Data_Quality_Flag'] = 'Zero Population'
df_api.loc[df_api['Area_Sq_Km'] == 0, 'Data_Quality_Flag'] = 'Zero Area'
df_api.loc[(df_api['Population'] == 0) & (df_api['Area_Sq_Km'] == 0), 'Data_Quality_Flag'] = 'Zero Pop & Area'
print(f"Step 3: Missing values filled, {(df_api['Data_Quality_Flag'] != 'OK').sum()} rows flagged for quality")

# Step 4: IQR outlier detection on Population and Area
def flag_iqr_outliers(series, label):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return (series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)

valid = df_api['Data_Quality_Flag'] == 'OK'
df_api['Population_Outlier'] = False
df_api['Area_Outlier'] = False
df_api.loc[valid, 'Population_Outlier'] = flag_iqr_outliers(df_api.loc[valid, 'Population'], 'Population').values
df_api.loc[valid, 'Area_Outlier'] = flag_iqr_outliers(df_api.loc[valid, 'Area_Sq_Km'], 'Area').values
print(f"Step 4: IQR outliers - {df_api['Population_Outlier'].sum()} population, {df_api['Area_Outlier'].sum()} area")

# Step 5: Remove duplicates
before = len(df_api)
df_api.drop_duplicates(subset='Country', keep='first', inplace=True)
df_api.reset_index(drop=True, inplace=True)
print(f"Step 5: Duplicates - {before - len(df_api)} removed, {len(df_api)} remain")

# Step 6: Fuzzy matching against WHR country names
whr_countries = df_whr['country'].tolist()
correction_map = {'Czechia': 'Czech Republic', 'Ivory Coast': 'Ivory Coast', 'TÃ¼rkiye': 'Turkey'}
no_exact = []
for api_name in df_api['Country'].tolist():
    if api_name not in whr_countries:
        result = process.extractOne(api_name, whr_countries, scorer=fuzz.token_sort_ratio)
        if result:
            best_match, score = result
            if score >= 80:
                correction_map[api_name] = best_match
                no_exact.append((api_name, best_match, score))

applied = 0
for old_name, new_name in correction_map.items():
    mask = df_api['Country'] == old_name
    if mask.any():
        df_api.loc[mask, 'Country'] = new_name
        applied += 1
print(f"Step 6: Fuzzy matching - {applied} country names corrected to match WHR")

# Step 7: Format for readability (calculated fields & rounding)
df_api['Pop_Density_Per_Sq_Km'] = np.where(
    df_api['Area_Sq_Km'] > 0,
    (df_api['Population'] / df_api['Area_Sq_Km']).round(2),
    np.nan)
df_api['Latitude'] = df_api['Latitude'].round(4)
df_api['Longitude'] = df_api['Longitude'].round(4)

def format_pop(val):
    if pd.isna(val) or val == 0: return 'N/A'
    elif val >= 1_000_000_000: return f"{val/1_000_000_000:.1f}B"
    elif val >= 1_000_000: return f"{val/1_000_000:.1f}M"
    elif val >= 1_000: return f"{val/1_000:.1f}K"
    else: return str(int(val))

df_api['Population_Formatted'] = df_api['Population'].apply(format_pop)
df_api.sort_values('Population', ascending=False, inplace=True)
df_api.reset_index(drop=True, inplace=True)
print(f"Step 7: Added Pop_Density, Population_Formatted, rounded coordinates")

print(f"\n\u2714 REST Countries API cleaned: {df_api.shape[0]} countries Ã— {df_api.shape[1]} columns")
print(f"  Columns: {list(df_api.columns)}")
df_api.head()


# Load Cleaned Datasets into SQLite Database

Each cleaned dataset is loaded as an individual table in SQLite. The database is saved to disk as quality_of_life.db so it persists after the notebook closes.

| SQLite Table | Source | Join Key | Rows |
|---|---|---|---|
| `happiness` | WHR CSV (Kaggle) | `country` | ~165 |
| `gdp` | Wikipedia HTML scrape | `Country_Territory` | ~58 |
| `countries` | REST Countries API | `Country` | ~250 |
**Below code creates/opens the SQLite database and writes each cleaned dataset into separate source tables.Persists cleaned data in a relational format so SQL joins and downstream analysis are repeatable.**

In [ ]:
# â”€â”€ Create SQLite database (file-based for persistence) â”€â”€
db_path = 'quality_of_life.db'
conn = sqlite3.connect(db_path)

# Load each dataset as a separate table
df_whr.to_sql('happiness', conn, if_exists='replace', index=False)
df_final_gdp.to_sql('gdp', conn, if_exists='replace', index=False)
df_api.to_sql('countries', conn, if_exists='replace', index=False)

# Verify tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"\u2714 SQLite database created: {db_path}")
print(f"  Tables loaded: {len(tables)}")
for _, row in tables.iterrows():
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {row['name']}", conn).iloc[0, 0]
    cols = pd.read_sql(f"PRAGMA table_info({row['name']})", conn)
    print(f"  â€¢ {row['name']}: {count} rows Ã— {len(cols)} columns")


#Join Datasets Using SQL
**The WHR `happiness` table is the base (left) table. I will now LEFT JOIN gdp on matching country names (happiness.country = gdp.Country_Territory) and LEFT JOIN countries on matching country names (happiness.country = countries.Country). This produces one consolidated dataset combining happiness scores, GDP data, and country metadata.**
**Below code runs a SQL query that LEFT JOINs happiness, GDP, and country metadata into one consolidated DataFrame. This process will combine all sources into a single analytical dataset while preserving happiness records as the base table.**

In [ ]:
# Join all three tables using SQL LEFT JOINs
join_query = """
SELECT 
    h.country,
    h.region,
    h.year,
    h.happiness_score,
    h.log_gdp_per_capita,
    h.social_support,
    h.healthy_life_expectancy,
    h.freedom_score,
    h.generosity,
    h.corruption_perception,
    h.positive_affect,
    h.negative_affect,
    h.is_outlier,
    g.GDP_Million_USD,
    g.GDP_Billion_USD,
    g.Economy_Category,
    g.GDP_Formatted,
    g.Rank AS gdp_rank,
    c.Capital_City,
    c.Continent,
    c.Subregion,
    c.Population,
    c.Area_Sq_Km,
    c.Pop_Density_Per_Sq_Km,
    c.Primary_Language,
    c.Language_Count,
    c.Currency_Name,
    c.Currency_Code,
    c.Latitude,
    c.Longitude,
    c.Population_Formatted
FROM happiness h
LEFT JOIN gdp g ON h.country = g.Country_Territory
LEFT JOIN countries c ON h.country = c.Country
ORDER BY h.happiness_score DESC
"""

df_merged = pd.read_sql(join_query, conn)

print(f"\u2714 Merged dataset: {df_merged.shape[0]} rows Ã— {df_merged.shape[1]} columns")
print(f"\nâ”€â”€ Join coverage â”€â”€")
print(f"  Rows with GDP data (WHR âˆ© Wikipedia): {df_merged['GDP_Billion_USD'].notna().sum()} / {len(df_merged)}")
print(f"  Rows with API data (WHR âˆ© REST API):  {df_merged['Population'].notna().sum()} / {len(df_merged)}")
both = df_merged.dropna(subset=['GDP_Billion_USD', 'Population']).shape[0]
print(f"  Rows with ALL 3 sources:              {both} / {len(df_merged)}")
print(f"\nColumns ({df_merged.shape[1]}): {list(df_merged.columns)}")
df_merged.head(10)


**Below code saves the merged DataFrame back into SQLite as a dedicated merged table. This is done to store the final integrated dataset for reuse without rerunning the full pipeline.**

In [ ]:
# Store merged dataset back into SQLite
df_merged.to_sql('merged', conn, if_exists='replace', index=False)
print("\u2714 Merged table stored in SQLite as 'merged'")
print(f"  Database file: {db_path}")

# Part 4: Visualizations

5 labeled visualizations are presented below. At least 2 use data from more than one source table via SQL joins.

| # | Title | Sources Used | Library |
|---|-------|-------------|---------|
| 1 | Top 20 Happiest Countries with GDP & Population | happiness + gdp + countries | Matplotlib/Seaborn |
| 2 | Happiness Score vs. GDP (Nominal) - Scatter | happiness + gdp | Plotly |
| 3 | Average Happiness by Region & Economy Category | happiness + gdp | Seaborn |
| 4 | Population Density vs. Social Support | happiness + countries | Plotly |
| 5 | Regional Happiness Distribution (Box Plot) | happiness | Seaborn |

### Visualization 1: Top 20 Happiest Countries with GDP and Population
**Sources:** happiness (CSV) + gdp (Wikipedia) + countries (API) - all three tables  
This horizontal bar chart shows the 20 countries with the highest happiness scores, annotated with their economy category and population from the other two data sources.
** Below code queries top countries by happiness and creates a labeled horizontal bar chart with GDP category and population annotations. This will provide a quick multi-source comparison of the happiest countries with economic and demographic context.

In [ ]:
#Visualization 1: Top 20 Happiest Countries (3-source join)
viz1_query = """
SELECT h.country, h.happiness_score, h.region,
       g.GDP_Billion_USD, g.Economy_Category,
       c.Population, c.Population_Formatted
FROM happiness h
LEFT JOIN gdp g ON h.country = g.Country_Territory
LEFT JOIN countries c ON h.country = c.Country
WHERE h.happiness_score IS NOT NULL
ORDER BY h.happiness_score DESC
LIMIT 20
"""
df_viz1 = pd.read_sql(viz1_query, conn)

fig, ax = plt.subplots(figsize=(14, 9))
colors = sns.color_palette('YlGnBu', n_colors=20)[::-1]
bars = ax.barh(range(len(df_viz1)), df_viz1['happiness_score'], color=colors)

ax.set_yticks(range(len(df_viz1)))
ax.set_yticklabels(df_viz1['country'], fontsize=10)
ax.set_xlabel('Happiness Score', fontsize=12)
ax.set_title('Visualization 1: Top 20 Happiest Countries\n(GDP Category from Wikipedia + Population from REST API)',
             fontsize=14, fontweight='bold', pad=15)
ax.invert_yaxis()

for i, (_, row) in enumerate(df_viz1.iterrows()):
    econ = row['Economy_Category'] if pd.notna(row['Economy_Category']) else 'N/A'
    pop = row['Population_Formatted'] if pd.notna(row['Population_Formatted']) else 'N/A'
    ax.text(row['happiness_score'] + 0.05, i, f"{econ} | Pop: {pop}",
            va='center', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('viz1_top20_happiness.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2714 Visualization 1 saved")


### Visualization 2: Happiness Score vs. GDP (Nominal) - Scatter Plot
**Sources:** happiness (CSV) + gdp (Wikipedia)**
**This interactive Plotly scatter plot explores whether wealthier countries report higher happiness scores, with a log-scaled x-axis colored by region.
I will now build an interactive scatter plot of happiness versus GDP using joined WHR and Wikipedia data. This is done to test and visualize the relationship between national wealth and reported happiness.**

In [ ]:
# -- Visualization 2: Happiness vs GDP (Plotly interactive) --
viz2_query = """
SELECT h.country, h.happiness_score, h.region,
       g.GDP_Billion_USD
FROM happiness h
INNER JOIN gdp g ON h.country = g.Country_Territory
WHERE g.GDP_Billion_USD IS NOT NULL AND h.happiness_score IS NOT NULL
"""
df_viz2 = pd.read_sql(viz2_query, conn)

fig2 = px.scatter(df_viz2, x='GDP_Billion_USD', y='happiness_score',
                  color='region', hover_name='country',
                  size=df_viz2['GDP_Billion_USD'].clip(upper=5000),
                  log_x=True,
                  labels={'GDP_Billion_USD': 'GDP Nominal (Billion USD, log scale)',
                          'happiness_score': 'Happiness Score',
                          'region': 'Region'},
                  title='Visualization 2: Happiness Score vs. GDP (Nominal)<br><sub>Sources: WHR CSV + Wikipedia GDP | Bubble size = GDP</sub>')
fig2.update_layout(height=600, width=950)

try:
    fig2.write_image('viz2_happiness_vs_gdp.png', scale=2)
    print("\u2714 Visualization 2 PNG saved")
except ValueError as e:
    print(f"PNG export skipped: {e}")

fig2.show()
print("\u2714 Visualization 2 displayed")

### Visualization 3: Average Happiness by Region & Economy Category (Heatmap)
**Sources:** happiness (CSV) + gdp (Wikipedia)  
This heatmap shows how average happiness varies across world regions and economy size categories, combining data from the flat file and the website source.
I will now aggregate average happiness by region and economy category and visualizes the result as a heatmap. This highlights cross-group patterns that are hard to see in raw country-level rows.

In [ ]:
#Visualization 3: Heatmap - Region Ã— Economy Category
viz3_query = """
SELECT h.region, g.Economy_Category, AVG(h.happiness_score) as avg_happiness,
       COUNT(*) as n_countries
FROM happiness h
INNER JOIN gdp g ON h.country = g.Country_Territory
WHERE h.region IS NOT NULL AND g.Economy_Category IS NOT NULL
GROUP BY h.region, g.Economy_Category
HAVING n_countries >= 1
"""
df_viz3 = pd.read_sql(viz3_query, conn)

pivot = df_viz3.pivot_table(values='avg_happiness', index='region',
                            columns='Economy_Category', aggfunc='mean')
cat_order = ['Small Economy (<$100B)', 'Medium Economy ($100B-$1T)', 'Large Economy (>$1T)']
pivot = pivot.reindex(columns=[c for c in cat_order if c in pivot.columns])

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=5.5,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Happiness Score'})
ax.set_title('Visualization 3: Average Happiness by Region & Economy Category\n(Sources: WHR CSV + Wikipedia GDP)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Economy Category (from Wikipedia GDP)', fontsize=11)
ax.set_ylabel('Region (from WHR)', fontsize=11)
plt.tight_layout()
plt.savefig('viz3_heatmap_region_economy.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2714 Visualization 3 displayed")

### Visualization 4: Population Density vs. Social Support - Scatter Plot
**Sources:** happiness (CSV) + countries (API)  
This interactive Plotly scatter investigates whether population density (from the REST Countries API) correlates with social support scores (from the WHR flat file).
**I will now create an interactive scatter plot comparing population density and social support by country and region. This will allow me to explores whether demographic crowding patterns align with perceived social support.**

In [ ]:
# Visualization 4: Pop Density vs Social Support (Plotly)
viz4_query = """
SELECT h.country, h.social_support, h.region,
       c.Pop_Density_Per_Sq_Km, c.Population
FROM happiness h
INNER JOIN countries c ON h.country = c.Country
WHERE c.Pop_Density_Per_Sq_Km IS NOT NULL
  AND h.social_support IS NOT NULL
  AND c.Pop_Density_Per_Sq_Km < 2000
"""
df_viz4 = pd.read_sql(viz4_query, conn)

fig4 = px.scatter(df_viz4, x='Pop_Density_Per_Sq_Km', y='social_support',
                  color='region', hover_name='country',
                  size='Population', size_max=30,
                  labels={'Pop_Density_Per_Sq_Km': 'Population Density (people/kmÂ²)',
                          'social_support': 'Social Support Score',
                          'region': 'Region'},
                  title='Visualization 4: Population Density vs. Social Support<br><sub>Sources: WHR CSV + REST Countries API | Bubble size = Population</sub>')
fig4.update_layout(height=600, width=950)

try:
    fig4.write_image('viz4_density_vs_support.png', scale=2)
    print("\u2714 Visualization 4 PNG saved")
except ValueError as e:
    print(f"PNG export skipped: {e}")

fig4.show()
print("\u2714 Visualization 4 displayed")

### Visualization 5: Regional Happiness Distribution (Box Plot)
**Source:** happiness (CSV)  
**This box plot shows the spread, median, and outliers of happiness scores across world regions, ordered by median happiness.**
**I will now build a regional happiness box plot and overlays individual country points. This will allow me to compare distributions across regions, revealing spread and potential outliers beyond central tendency.**

In [ ]:
# Visualization 5: Box Plot - Regional Happiness Distribution
viz5_query = """
SELECT country, happiness_score, region
FROM happiness
WHERE region IS NOT NULL AND region != 'Unclassified'
ORDER BY region, happiness_score
"""
df_viz5 = pd.read_sql(viz5_query, conn)

region_order = (df_viz5.groupby('region')['happiness_score']
                .median().sort_values(ascending=False).index.tolist())

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=df_viz5, x='region', y='happiness_score', order=region_order,
            palette='Set2', ax=ax, linewidth=1.2)
sns.stripplot(data=df_viz5, x='region', y='happiness_score', order=region_order,
              color='black', alpha=0.4, size=4, jitter=True, ax=ax)

ax.set_title('Visualization 5: Happiness Score Distribution by Region\n(Source: WHR CSV  happiness table)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Region', fontsize=12)
ax.set_ylabel('Happiness Score', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('viz5_regional_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2714 Visualization 5 saved")

# Part 5: Final Merged Dataset Human Readable Output
**The complete merged dataset after all transformations and SQL joins. This combines data from all three sources: WHR flat file (Kaggle CSV), Wikipedia GDP (HTML scraping), and REST Countries API (JSON).**
**I will now read the merged table and prints a full human-readable dataset with summary statistics. This will provide transparent final output for inspection, validation, and reporting of integration results.**

In [ ]:
# Display the full merged dataset in a notebook-friendly format
from IPython.display import display
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df_final = pd.read_sql("SELECT * FROM merged ORDER BY happiness_score DESC", conn)

# Save Part 5 output to JSON for reuse/sharing
milestone5_json_path = 'milestone5_final_merged_output.json'
df_final.to_json(milestone5_json_path, orient='records', indent=2, force_ascii=False)
print(f"Saved JSON output: {milestone5_json_path}")

print("=" * 100)
print("GLOBAL QUALITY OF LIFE - FINAL MERGED DATASET | Milestone 5")
print(f"{df_final.shape[0]} countries x {df_final.shape[1]} columns")
print("Sources: WHR (CSV) + Wikipedia GDP (HTML) + REST Countries API (JSON)")
print("=" * 100)

# Render as table in notebook output
display(df_final)

# Summary statistics
print(f"\n{'='*80}")
print("SUMMARY STATISTICS")
print(f"{'='*80}")
print(f"Total countries: {len(df_final)}")
print(f"Countries with GDP data: {df_final['GDP_Billion_USD'].notna().sum()}")
print(f"Countries with API data: {df_final['Population'].notna().sum()}")
print(f"Countries with all 3 sources: {df_final.dropna(subset=['GDP_Billion_USD', 'Population']).shape[0]}")
print(f"\nHappiness Score range: {df_final['happiness_score'].min():.3f} - {df_final['happiness_score'].max():.3f}")
gdp_valid = df_final['GDP_Billion_USD'].dropna()
if len(gdp_valid) > 0:
    print(f"GDP range: ${gdp_valid.min():.2f}B - ${gdp_valid.max():.2f}B")
pop_valid = df_final['Population'].dropna()
if len(pop_valid) > 0:
    print(f"Population range: {pop_valid.min():,.0f} - {pop_valid.max():,.0f}")

# Restore default row display limit
pd.set_option('display.max_rows', 20)

**I will now close the SQLite connection and prints completion confirmation messages. This process will releases database resources cleanly and signals that the notebook workflow finished successfully.**

In [ ]:
#Close database connection
conn.close()
print("\u2714 SQLite connection closed.")
print(f"  Database saved to: {db_path}")
print("\n\u2714 Milestone 5 complete.")

**Part 6: Project Summary & Ethical Implications**

Completing this project required pulling three fundamentally different data sources. The three data sources are Kaggle CSV, a scraped Wikipedia HTML table, and a live REST API. I have cleaned each data source independently across Milestones 2 through 4, and then merged them into a single consolidated dataset inside a SQLite database. The biggest technical challenge was getting country names to align across all three sources. Each dataset used slightly different conventions in the World Happiness Report(WHR) file had Turkiye with a special character, Wikipedia used bracketed footnotes like China[n 1], and the REST Countries API returned Czechia where the happiness data used Czech Republic. Fuzzy matching with the thefuzz library solved most mismatches automatically, but some required manual correction maps. I learned that SQL joins are only as good as the join key. Even one mismatched name means a country silently drops out of the merged result, so verifying join coverage after the merge was essential. Combining Matplotlib/Seaborn for static charts with Plotly for interactive scatter plots gave me flexibility to choose the right tool for each question. The heatmap crossing regions with economy categories was particularly revealing: happiness does not scale linearly with GDP, and Western European countries in the Medium Economy bracket often scored higher than large economies in other regions.

The transformations I applied included renaming headers to snake_case, stripping non-ASCII characters and footnote annotations, converting comma-formatted GDP strings to numeric values, imputing missing WHR scores using per-country then regional median fallback, flagging IQR outliers without removing them, flattening nested API JSON into tabular form, and adding calculated fields like GDP in billions and population density. These changes carry ethical weight. Imputing with regional medians assumes countries within a region share similar wellbeing characteristics, which risks erasing real differences. For example, a stable democracy and a conflict-affected state sharing a geographic label. Filtering to the most recent survey year introduces temporal inconsistency, Finland reports from 2023 while South Sudan reports from 2017, yet they appear side by side. The IQR outlier flag on Afghanistan's extremely low score could be misread as a data error when it reflects a genuine humanitarian crisis. Legally, all three sources are openly licensed: the WHR is published by the UN for public use, Wikipedia falls under Creative Commons CC BY-SA, and the REST Countries API is open-source. No personally identifiable information is involved. Credibility was verified by tracing each source to its upstream authority such as Gallup World Poll, IMF World Economic Outlook, and UN/ISO standards respectively. All data was acquired ethically through public downloads, a single web request with a proper User-Agent header, and a standard API call with no authentication bypass. To mitigate these risks I preserved raw data alongside cleaned versions, documented every assumption in labeled notebook cells, logged fuzzy match corrections with confidence scores, and maintained original names in traceability columns so any data analyst can audit my decisions.